<a href="https://colab.research.google.com/github/Nafis28/Simple-Scrape/blob/main/LinkScrape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
pip install cloudscraper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 3.4 MB/s eta 0:00:00


In [9]:
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin, urlparse

def scrape_links_to_excel(url, output_filename="links.xlsx"):
    print(f"Scraping links from: {url}...")

    try:
        scraper = cloudscraper.create_scraper(browser={
            'browser': 'chrome',
            'platform': 'windows',
            'desktop': True
        })
        response = scraper.get(url, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"Error fetching the URL: {e}")
        return

    soup = BeautifulSoup(response.content, 'html.parser')

    # A list of file extensions we want to ignore
    ignored_extensions = (
        '.png', '.jpg', '.jpeg', '.gif', '.svg', '.webp',  # Images
        '.css', '.js', '.json',                            # Code files
        '.woff', '.woff2', '.ttf', '.eot',                 # Font files
        '.pdf', '.doc', '.docx', '.zip'                    # Documents/Downloads
    )

    links_data = []
    seen_urls = set() # We use a Set to keep track of links we've already seen

    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href'].strip()
        text = a_tag.get_text(strip=True)

        # 1. Filter out non-webpage protocols and page-jump anchors
        if href.startswith(('tel:', 'mailto:', 'javascript:', '#')):
            continue

        full_url = urljoin(url, href)
        parsed_url = urlparse(full_url)

        # 2. Filter out anything that isn't standard HTTP/HTTPS
        if parsed_url.scheme not in ['http', 'https']:
            continue

        # 3. Filter out direct links to static files (like fonts or images)
        if parsed_url.path.lower().endswith(ignored_extensions):
            continue

        # 4. Add to our list only if we haven't seen this exact URL yet
        if full_url not in seen_urls:
            seen_urls.add(full_url)
            links_data.append({
                'Link Text': text,
                'URL': full_url
            })

    # Export the data to an Excel file
    if links_data:
        df = pd.DataFrame(links_data)
        try:
            df.to_excel(output_filename, index=False)
            print(f"Success! {len(links_data)} unique page links saved to '{output_filename}'")
        except Exception as e:
            print(f"Error saving the Excel file: {e}")
    else:
        print("No valid page links were found on this webpage.")

# --- Run the Script ---
if __name__ == "__main__":
    target_website = "https://vanillebistro.com.au"
    scrape_links_to_excel(target_website, "vanillebistro_links.xlsx")

Scraping links from: https://vanillebistro.com.au...
Success! 11 unique page links saved to 'vanillebistro_links.xlsx'
